# DS605 Lab 5 — Machine Learning with Scikit-learn and From Scratch

This notebook completes Parts A, B, and C of the assignment using the UCI **Productivity Prediction of Garment Employees** dataset.

- Regression target: `actual_productivity`
- Classification target: `MeetsTarget = 1` when `actual_productivity >= targeted_productivity`, else `0`
- The classification feature set explicitly excludes `actual_productivity`.
- Both implementations reuse exactly the same train/test row indices.
- The manual implementation uses only NumPy and Pandas for preprocessing, model fitting, prediction, and metrics. Scikit-learn is used only in Part A.

## Dataset setup
Download the UCI dataset CSV and save it as:

`../data/garments_worker_productivity.csv`

The notebook also checks common alternative CSV filenames in the `data/` folder. The dataset is not bundled with this project, so results and comparison tables are generated when you run the notebook with the real CSV.

## 1. Imports, paths, and dataset loading

In [ ]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
ROOT = Path("..")
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
for folder in [TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

candidate_files = [
    DATA_DIR / "garments_worker_productivity.csv",
    DATA_DIR / "garment_worker_productivity.csv",
    DATA_DIR / "garments_worker_productivity.csv",
    DATA_DIR / "garments_worker_productivity.xlsx"
]
data_path = next((p for p in candidate_files if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Download the UCI Garment Employee Productivity CSV "
        "and place it at ../data/garments_worker_productivity.csv"
    )

if data_path.suffix.lower() in [".xlsx", ".xls"]:
    raw_df = pd.read_excel(data_path)
else:
    raw_df = pd.read_csv(data_path)

print("Dataset file:", data_path.name)
print("Shape:", raw_df.shape)
display(raw_df.head())
print("\nColumns:", raw_df.columns.tolist())
print("\nMissing values:\n", raw_df.isna().sum())

## 2. Data cleaning, target creation, and feature definitions

In [ ]:
# Standardize column names for reliable handling.
df = raw_df.copy()
df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

required = {"actual_productivity", "targeted_productivity"}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}")

# Convert numeric-looking columns where possible; preserve genuine categorical columns.
for col in df.columns:
    if col not in {"date", "quarter", "department", "day"}:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["actual_productivity", "targeted_productivity"]).reset_index(drop=True)
df["meets_target"] = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int)

# Exclude identifiers/date from predictors; retain production-related fields.
excluded = {"actual_productivity", "meets_target", "date"}
all_feature_cols = [c for c in df.columns if c not in excluded]
# The classification task must never use actual_productivity as an input.
classification_feature_cols = [c for c in all_feature_cols if c != "actual_productivity"]
regression_feature_cols = [c for c in all_feature_cols if c != "meets_target"]

# Identify numeric/categorical columns from training data later to avoid leakage.
y_reg = df["actual_productivity"].astype(float).to_numpy()
y_cls = df["meets_target"].astype(int).to_numpy()

print("Rows after removing missing targets:", len(df))
print("\nMeetsTarget distribution:")
print(df["meets_target"].value_counts().rename(index={0: "Below target", 1: "Meets target"}))
print("\nRegression features:", regression_feature_cols)
print("\nClassification features:", classification_feature_cols)
assert "actual_productivity" not in classification_feature_cols

## 3. One fixed train/test split shared by every model

A single deterministic split is generated once using NumPy. Both Scikit-learn and the manual NumPy/Pandas implementation use these exact row indices. This avoids comparing models trained and tested on different samples.

In [ ]:
def fixed_split_indices(n_rows, test_size=0.20, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(n_rows)
    n_test = int(np.ceil(n_rows * test_size))
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]
    return train_idx, test_idx

train_idx, test_idx = fixed_split_indices(len(df))
print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Same split will be reused in every section.")

# Part A — Scikit-learn implementation

The preprocessing pipeline imputes missing numeric values with the median, imputes categorical values with the most frequent category, one-hot encodes categorical features, and standardizes numeric features. Pipelines are fitted on training data only.

In [ ]:
def split_xy(frame, feature_columns, target):
    X = frame[feature_columns].copy()
    y = frame[target].to_numpy()
    return X.iloc[train_idx], X.iloc[test_idx], y[train_idx], y[test_idx]

def make_sklearn_preprocessor(X_train):
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    # Compatibility with both newer and older scikit-learn versions.
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", encoder)
    ])
    transformer = ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", categorical_pipe, categorical_cols)
    ], remainder="drop")
    return transformer

def sklearn_fit_evaluate(task, model_name, estimator, X_train, X_test, y_train, y_test):
    preprocessor = make_sklearn_preprocessor(X_train)
    model = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - t0
    t0 = time.perf_counter()
    pred = model.predict(X_test)
    predict_time = time.perf_counter() - t0

    if task == "regression":
        rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
        metrics = {
            "task": task, "implementation": "Scikit-learn", "model": model_name,
            "MAE": mean_absolute_error(y_test, pred), "RMSE": rmse,
            "R2": r2_score(y_test, pred),
            "training_time_sec": train_time, "prediction_time_sec": predict_time
        }
    else:
        metrics = {
            "task": task, "implementation": "Scikit-learn", "model": model_name,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "F1": f1_score(y_test, pred, zero_division=0),
            "training_time_sec": train_time, "prediction_time_sec": predict_time
        }
        print("\n", model_name, "classification report:\n",
              classification_report(y_test, pred, zero_division=0))
        print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    return model, metrics, pred

# Regression: actual_productivity is the target.
Xr_train, Xr_test, yr_train, yr_test = split_xy(df, regression_feature_cols, "actual_productivity")
sk_lr, sk_lr_result, sk_lr_pred = sklearn_fit_evaluate(
    "regression", "LinearRegression", LinearRegression(),
    Xr_train, Xr_test, yr_train, yr_test
)

# Classification: actual_productivity is NOT included in X.
Xc_train, Xc_test, yc_train, yc_test = split_xy(df, classification_feature_cols, "meets_target")
assert "actual_productivity" not in Xc_train.columns
sk_log, sk_log_result, sk_log_pred = sklearn_fit_evaluate(
    "classification", "LogisticRegression",
    LogisticRegression(max_iter=3000, class_weight=None, random_state=RANDOM_STATE),
    Xc_train, Xc_test, yc_train, yc_test
)

sklearn_results = pd.DataFrame([sk_lr_result, sk_log_result])
display(sklearn_results)
sklearn_results.to_csv(TABLE_DIR / "sklearn_results.csv", index=False)

# Part B — From-scratch implementation using NumPy and Pandas only

The manual preprocessing below is fitted only on training rows. Numeric missing values are replaced with training medians and standardized using training means and standard deviations. Categorical missing values are replaced with training modes and one-hot encoded using categories learned from the training set. Unknown test categories become all-zero columns.

The manual models and metrics use vectorized NumPy operations. No Scikit-learn preprocessing, model, metric, or splitting utility is called in this section.

In [ ]:
# Manual preprocessing: Pandas + NumPy only.
class ManualPreprocessor:
    def __init__(self):
        self.numeric_cols = None
        self.categorical_cols = None
        self.medians = {}
        self.means = {}
        self.stds = {}
        self.modes = {}
        self.categories = {}

    def fit(self, X):
        self.numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols = [c for c in X.columns if c not in self.numeric_cols]
        for col in self.numeric_cols:
            values = pd.to_numeric(X[col], errors="coerce")
            med = values.median()
            if pd.isna(med): med = 0.0
            filled = values.fillna(med).astype(float)
            mean = float(filled.mean())
            std = float(filled.std(ddof=0))
            if not np.isfinite(std) or std == 0: std = 1.0
            self.medians[col], self.means[col], self.stds[col] = float(med), mean, std
        for col in self.categorical_cols:
            values = X[col].astype("object")
            mode_values = values.dropna().mode()
            mode = str(mode_values.iloc[0]) if len(mode_values) else "__missing__"
            self.modes[col] = mode
            filled = values.fillna(mode).astype(str)
            self.categories[col] = sorted(filled.unique().tolist())
        return self

    def transform(self, X):
        blocks = []
        if self.numeric_cols:
            numeric = []
            for col in self.numeric_cols:
                values = pd.to_numeric(X[col], errors="coerce").fillna(self.medians[col]).to_numpy(dtype=float)
                scaled = (values - self.means[col]) / self.stds[col]
                numeric.append(scaled)
            blocks.append(np.column_stack(numeric))
        for col in self.categorical_cols:
            values = X[col].fillna(self.modes[col]).astype(str).to_numpy()
            cats = self.categories[col]
            onehot = np.zeros((len(X), len(cats)), dtype=float)
            cat_to_idx = {cat: j for j, cat in enumerate(cats)}
            for i, value in enumerate(values):
                j = cat_to_idx.get(value)
                if j is not None:
                    onehot[i, j] = 1.0
            blocks.append(onehot)
        if not blocks:
            return np.empty((len(X), 0), dtype=float)
        return np.concatenate(blocks, axis=1)

def manual_prepare(X_train, X_test):
    prep = ManualPreprocessor().fit(X_train)
    return prep, prep.transform(X_train), prep.transform(X_test)

# Use the same train/test rows as Part A.
prep_reg, Xr_train_manual, Xr_test_manual = manual_prepare(Xr_train, Xr_test)
prep_cls, Xc_train_manual, Xc_test_manual = manual_prepare(Xc_train, Xc_test)
print("Manual regression matrix:", Xr_train_manual.shape, Xr_test_manual.shape)
print("Manual classification matrix:", Xc_train_manual.shape, Xc_test_manual.shape)

In [ ]:
# Manual Linear Regression (least-squares solution with intercept).
class ManualLinearRegression:
    def __init__(self):
        self.weights = None

    def fit(self, X, y):
        X_bias = np.column_stack([np.ones(X.shape[0]), X])
        self.weights = np.linalg.lstsq(X_bias, y.astype(float), rcond=None)[0]
        return self

    def predict(self, X):
        X_bias = np.column_stack([np.ones(X.shape[0]), X])
        return X_bias @ self.weights

def manual_regression_metrics(y_true, y_pred):
    errors = y_true - y_pred
    mae = float(np.mean(np.abs(errors)))
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    ss_res = float(np.sum(errors ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

t0 = time.perf_counter()
manual_lr = ManualLinearRegression().fit(Xr_train_manual, yr_train)
manual_lr_train_time = time.perf_counter() - t0
t0 = time.perf_counter()
manual_lr_pred = manual_lr.predict(Xr_test_manual)
manual_lr_predict_time = time.perf_counter() - t0
manual_lr_metrics = manual_regression_metrics(yr_test, manual_lr_pred)
manual_lr_result = {
    "task": "regression", "implementation": "Manual NumPy/Pandas",
    "model": "Manual Linear Regression", **manual_lr_metrics,
    "training_time_sec": manual_lr_train_time,
    "prediction_time_sec": manual_lr_predict_time
}
print(manual_lr_result)

In [ ]:
# Manual Logistic Regression with sigmoid and gradient descent.
class ManualLogisticRegression:
    def __init__(self, learning_rate=0.05, epochs=6000, l2=0.0, tolerance=1e-8):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.l2 = l2
        self.tolerance = tolerance
        self.weights = None
        self.loss_history = []

    @staticmethod
    def sigmoid(z):
        z = np.clip(z, -35, 35)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        Xb = np.column_stack([np.ones(X.shape[0]), X])
        y = y.astype(float)
        self.weights = np.zeros(Xb.shape[1], dtype=float)
        n = Xb.shape[0]
        previous_loss = np.inf
        for epoch in range(self.epochs):
            probabilities = self.sigmoid(Xb @ self.weights)
            errors = probabilities - y
            gradient = (Xb.T @ errors) / n
            # Do not regularize the intercept.
            gradient[1:] += (self.l2 / n) * self.weights[1:]
            self.weights -= self.learning_rate * gradient

            if epoch % 100 == 0 or epoch == self.epochs - 1:
                p = np.clip(self.sigmoid(Xb @ self.weights), 1e-12, 1 - 1e-12)
                loss = -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
                loss += (self.l2/(2*n)) * np.sum(self.weights[1:]**2)
                self.loss_history.append(float(loss))
                if abs(previous_loss - loss) < self.tolerance:
                    break
                previous_loss = loss
        return self

    def predict_proba(self, X):
        Xb = np.column_stack([np.ones(X.shape[0]), X])
        return self.sigmoid(Xb @ self.weights)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

def manual_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-15)
    return {"accuracy": float(accuracy), "precision": float(precision),
            "recall": float(recall), "F1": float(f1),
            "TP": tp, "TN": tn, "FP": fp, "FN": fn}

t0 = time.perf_counter()
manual_log = ManualLogisticRegression(learning_rate=0.05, epochs=6000, l2=0.0).fit(
    Xc_train_manual, yc_train
)
manual_log_train_time = time.perf_counter() - t0
t0 = time.perf_counter()
manual_log_pred = manual_log.predict(Xc_test_manual)
manual_log_predict_time = time.perf_counter() - t0
manual_log_metrics = manual_classification_metrics(yc_test, manual_log_pred)
manual_log_result = {
    "task": "classification", "implementation": "Manual NumPy/Pandas",
    "model": "Manual Logistic Regression", **manual_log_metrics,
    "training_time_sec": manual_log_train_time,
    "prediction_time_sec": manual_log_predict_time
}
print(manual_log_result)
print("Manual logistic confusion matrix [[TN, FP], [FN, TP]]:")
print(np.array([[manual_log_metrics["TN"], manual_log_metrics["FP"]],
                [manual_log_metrics["FN"], manual_log_metrics["TP"]]]))

# Part C — Comparison and optimization

The first comparison puts Scikit-learn and manual results side by side. The optimization then trains a second manual logistic regression with L2 regularization and a smaller learning rate. This can reduce overfitting and improve numerical stability; the final result is judged using the held-out test metrics and runtime, not by assuming it must outperform the baseline.

In [ ]:
manual_results = pd.DataFrame([manual_lr_result, manual_log_result])
all_results = pd.concat([sklearn_results, manual_results], ignore_index=True, sort=False)
display(all_results)
all_results.to_csv(TABLE_DIR / "sklearn_vs_manual_comparison.csv", index=False)

In [ ]:
# Optimized manual logistic regression: L2 regularization + tuned learning rate/iterations.
t0 = time.perf_counter()
optimized_log = ManualLogisticRegression(
    learning_rate=0.03, epochs=10000, l2=0.05, tolerance=1e-9
).fit(Xc_train_manual, yc_train)
optimized_train_time = time.perf_counter() - t0
t0 = time.perf_counter()
optimized_pred = optimized_log.predict(Xc_test_manual)
optimized_predict_time = time.perf_counter() - t0
optimized_metrics = manual_classification_metrics(yc_test, optimized_pred)
optimized_result = {
    "task": "classification", "implementation": "Manual NumPy/Pandas (optimized)",
    "model": "Manual Logistic Regression + L2",
    **optimized_metrics,
    "training_time_sec": optimized_train_time,
    "prediction_time_sec": optimized_predict_time
}
optimization_comparison = pd.DataFrame([manual_log_result, optimized_result, sk_log_result])
display(optimization_comparison)
optimization_comparison.to_csv(TABLE_DIR / "optimization_comparison.csv", index=False)

plt.figure(figsize=(7,4))
plt.plot(np.arange(len(optimized_log.loss_history))*100, optimized_log.loss_history)
plt.xlabel("Training epoch")
plt.ylabel("Regularized log-loss")
plt.title("Optimized manual logistic regression convergence")
plt.tight_layout()
plt.savefig(FIG_DIR / "optimized_logistic_loss.png", dpi=160)
plt.show()

## 6. Final observations and saved outputs

In [ ]:
def fmt(v):
    return f"{v:.4f}" if isinstance(v, (float, np.floating)) else str(v)

obs = f'''
# Lab 5 — Experimental observations

## Dataset
- Rows used after target cleaning: {len(df)}
- Training rows: {len(train_idx)}
- Test rows: {len(test_idx)}
- MeetsTarget class counts: {df["meets_target"].value_counts().to_dict()}

## Regression
- Scikit-learn Linear Regression: MAE={sk_lr_result["MAE"]:.5f}, RMSE={sk_lr_result["RMSE"]:.5f}, R2={sk_lr_result["R2"]:.5f}
- Manual Linear Regression: MAE={manual_lr_result["MAE"]:.5f}, RMSE={manual_lr_result["RMSE"]:.5f}, R2={manual_lr_result["R2"]:.5f}

## Classification
- Scikit-learn Logistic Regression: accuracy={sk_log_result["accuracy"]:.5f}, precision={sk_log_result["precision"]:.5f}, recall={sk_log_result["recall"]:.5f}, F1={sk_log_result["F1"]:.5f}
- Manual Logistic Regression: accuracy={manual_log_result["accuracy"]:.5f}, precision={manual_log_result["precision"]:.5f}, recall={manual_log_result["recall"]:.5f}, F1={manual_log_result["F1"]:.5f}
- Optimized manual Logistic Regression (L2): accuracy={optimized_result["accuracy"]:.5f}, precision={optimized_result["precision"]:.5f}, recall={optimized_result["recall"]:.5f}, F1={optimized_result["F1"]:.5f}

## Interpretation
- The same fixed row indices were reused in every implementation.
- Regression uses MAE, RMSE, and R2; classification uses accuracy, precision, recall, and F1.
- Manual linear regression uses a least-squares solution. Manual logistic regression uses a sigmoid function and vectorized gradient descent.
- The optimized manual logistic model adds L2 regularization and adjusts the learning rate/iteration budget.
- Scikit-learn may run faster because its numerical routines are highly optimized. Manual code is valuable for demonstrating the underlying workflow, not necessarily for beating a mature library's runtime.
- Compare the generated timing columns and held-out metrics before drawing a conclusion about which implementation is preferable.
'''
(OUT_DIR / "OBSERVATIONS.md").write_text(obs, encoding="utf-8")
print(obs)
print("\nSaved output files:")
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(path)